# ACE F1 — Baseline Harness (reproduksi CPU)

**Penting:** notebook ini menguji solver tetap A-Fixed dan infrastruktur eksperimen. Tidak menjalankan evolusi AI, tidak membuka data held-out, tidak menghubungkan API, tidak mengeksekusi kode AI tak tepercaya.

Di HP: **Runtime → Run all**. Hasil akhir yang diharapkan: `F1_LOCAL_PASS`, **14/14 tes**, **108 percobaan**, **83 jawaban benar**. Notebook akan menawarkan file `ACE_F1_reproduced_evidence.zip` untuk Anda unggah kembali ke ChatGPT.

Folder `/content` bersifat sementara. Unduh ZIP agar bukti tidak hilang saat runtime berhenti.


In [ ]:
# ACE F1 self-contained no API/no installation notebook. Trusted local baseline only.
import os, base64, io, json, zipfile, subprocess, sys, pathlib, platform, datetime, hashlib
payload = "UEsDBBQAAAAIAJQgZF1cx3o3fyEAAEprAAAJAAAAYWNlX2YxLnB51T3tcttGkv/1FIhTWwAskCJtr3cDB6mSZdnRxbFdlpy7LR4LCxJDERYJwABISVZUdQ9xT3hPct093wAoya69H+eqRAIw09PT09/TM3r06NHh0bHzeuykrGHVOsuzusnmziK7YqkzS2q2ynLmLJMqZ3XteHWTrrKZU+Sra3+4t3c6L0oWOk21qRtov0zydJBsmmVRwVOyOi+qrFmua2gP8LdsVZRrljcH22SVpUmTwWsYqNlUrCaQw71f2SodFJvGaZL6ooZOMBqrkoatrp20cPKicdgVoOhkudMss9opk/lFcs6Ge48ePdpbVMXaiePFBkHGsZOty6JqnCSHfjRcvbcn31XnZVLVTD4vk3oJY6lHlpRf5MPnusjl70Utf6tgssVaPtXLTZOt1NO1atawdbnIVmqgJlszjui8WK3YnNCSmB4VmxyWgX8HCjFsLT/K54BgfC1yAadMGkRdNvsAj3t7L4/fHf36++HH3+I/jj+enrx/50SOm8zZYMby+XIwGo6HI3fv9P1b+Gw2ORzQ0ssGx38cvv10ePb+YxsMgzXcJE1RKVBnn179Iz55RUCOjgevx4MnoyfPx+PRM/h4fPzqFL544789+Slw4MPfAufZ6Ken/t7HT+/is5Pfj99/OotPj4/ev6OG47+PRnuHb9++//f4wyEBfZ2sYLVO//H7y/dvT47iV8d/IDy3AraCZXQDx61wkVm8YosGH4HoQMqYfcGHeQHEgWa+hgAzIwgpSzclAagvkzIuk6yqod2bj4cfflXDXLLsfAksHm/yNKtg1ViKPdRr8+V5laVxDTLQsLpRkORwaxCVeJvVGSG5Tq7iZVHigHvzVQIidgIMABCa6+OqKirvIzAErDY9+OGeA/9KaCZbv9yk56x5UNPD+RxE+BXLM5Z6H1DW6xp4r9t8L2ULZ57kRZ7Nk5VXzD6LzxUDucpJHIbpZl3W+C1waphpfMGu6+is2gB3srxG8UvqeZZFtGzQhoG0IbvUkecGOPPQ9QNQEaviMs6TnLfzxeBpdg6kM0bOFg5QLK+bJJ8zPursGsgrPuO/GVAXPtAzaI1uh7qpus2HIA1Fyjx30yz+DovAe9fMbmgTo6+PII3QIsN6mTz563Nv5g+X7ErMRk4OqLDO5jFS0UPRDUliA0fPFt8OgV6gKofrC2Atjz8oAqMGjIsLeuTjN+sS8KSOl6BygaRrQHDoOvv8JT7D7+4QGgqUm+pazxI7OaDLcw8aIGO7uI4wzSw/j8REnaR2FroL/lsML4FVmWcTCAf6z1wMo5suVpt66dlvi3q4qK/zuQefQUnmhefrBvCxYuUK1A3HCmfCvy6yHFjHwB/WG5oMiTK159tY4pdNDobsQi1CxZJUL4For4lQ0orcQwBTHFZFknoLCR2nEiMvGAtsy5DgCVobwoXYGSfPIcRoXbyasTRwFsk6W10DA5errAE+qVKcvQ2PG6PhR/rhZXnjiREW7g1Cuf3zhoOBXwgO/BSAbl1/Eo6fTwNn/FyPj/bX6x85IJu2Al0LS1IVsxVbgyTn9SVo20VRrZMG5Z2lipvBKEXOjWt4AG7ouKBaUQ9oXwBfgpZ0byc03tSc342iuouYxVkKrWFuYlKDGxwFfggUw9Gz9JaMAH3HtnwqGgyNAR/E3Fw5pzbo4Y38cjvcjl0DAlnSdVJdxGiC+AQ6ZhfNCtACB8LF1L0F5eCDoqFrERG+2ETVfVfZOmtqaHBDBgT8pXLTxCCIa3z5/JkwLMhdYA7z82YJr58+uSUIt2KJz1lOnpVnrBU4UW/Ea2f8xPTZQKafO4bfdvKqdh6XrBKEfRyAyyRcNzBDxWa+ZOkQfTISQHLmImfC17QpLlgel0WxgncrkFjPTZLZPGVCZcB85bJI2tcBuEsAOMYhwfebeBY7BY7pGATOU7AtnslaRgNgMWww1YKM413osRA+yzdrTh2FQUupYKfP2BRE7xw0oMSu1Yz0GGdKmOyF89iYxz4A2HfGnfZVfg5tTRXgnrod+e90I7ISmQHAcL4sMlScitY+oRxrlLEVjoHaYhQ4P/n+tAMT9aokSwSenXS3QtCTNNSNW5QouPLDbRcGWWMbiuGpdSHZbhwEFiVydQvb8TP/QUNpL7AzkOUgojvLxDBd0j1oKOlj7qZN4PzfTFK4sJ2BxfsHwTCc3w4c41svLPCXgKeymjmHdc2qRrqVSnh6mBVVwjApwdqmHrc2PUxumJoOBPx343KmByT5L0BLMOAVD/b4RG5RudPHOKmq5NoV1un7dY0KDSCG6Soa5e7//9Mybx6gZdp8Y0c7Ye8yzVbF/IKlyFKTMbA1+BqTJ/hziuA+O39xniCsEbESb/NEtHkynfaCFEaTHIuLLEcDS6jwQPAS1/4ZxX0r+atAAp7Eb/0sZfwDsQSvAjpMQBZHUwztimSFz09hbae3/ZLQCxX9bxIp9xAxfIn/O8L/vcL/Hbv9k2QQ3FGvnZga8Fq6A3gvuKPbSz6u3Q3o/te7ux1KxFvd7hlNTfXbkHy1A8mHzK0HyWe7uvVTv83p3WA/3IkEX7l9ZHg++fYsgLX/5n8TbyflEgHlEHoiSxNHwTONBM/0836W1rjfN7mHiwfnCiEbMNPbB5HTSIQoH3hivp1iKumV2y8ZHfulMikmMPkOQT15kA164z4k3LmPNNzDR/ebovF5UTe22REWg9Go8TrJswVGa4SRlVGgN8Jjv7OTSJSkqC9q1njavPFkKTay4+WJCqWm2AK6Siv+BzpD3IK74EGsILIHKmNr8PuNuB66DJM09UxQfmsMHmdNKXULo7RtqWk7W0ZNeBRm2spVQQaBxcQMwp0xFc2kOMjrsdtGQwSCBh7obcBy+7DO/KXOF0F7GZ6B1k+zedOLmkmmLKeJEJFkHumqJElCORbDmXP3QwixAomIRQbneZfDOGv2Q3jTA4ELYTJvNuQTiMyy5xm0CIz18Vus4su8m4DwQ6Tm08MlC1dyI3c8gJNueMdbI6aDcW3Sa6LCtJBMyEhGHD5tIdVayyhCiKBQjXlEkUUirYq2RIZ/zQgGqW9NPkM2gtGHWZ1m9ecC9fwWMzU9/KJ014oluIFh5xExsSeEvtzMQP64dpJs+CM0SPJ6Ad5ds6yKzfnS+bfT9+/ABwYNBbOHcH294TseTgEgqmyrBLhuqs0c90eGe31JrNrI5d1cgJGYXHAaXfClk3KucyuBzmV0UhiBylX4tyq3dM4aPh3pd5v6ayEkG1W6FHY37NUFr58NcNMo1JmHrKYlABrUSANTEyjAD9JDPQu2yS/y4jLnUOzVmthr1M9SintEcovTIkOhBHybuPZUHqjelLiTw9LoXZEzQRkV3SvrJkKf6SSc7hkMaOoxERNhjkXpuWS18ow2VzwzTkhfEdLUyZDNHubluKCtRt0L/bP8vDYkvSgRksLUCMmMgIhy0hG0xQaGB4xLJSkglxQJQXJIncQaqlYt5cxXBbtYwknDWTkMRVP+yyQMB2ONB/kYupeZs7DGq8UkeBA/hWhmhXl0TkZafT4MBTejvd6UjUCgDqcQq0ls6l24mEkNnfW5aq8hjn2FHRA7nuCY7gKpUggKHiXm0PYNcZcR93fknPwdMMwUQieKzXQUC46/QaEBRnu+HHiSoVkSv+6Pp5o2+KQ/mfPQ6Yc+gVXMZwstByTEsClicrClDGptpHiYHPGpCnmNGWK4iSmDlSWdFIROA/2CQlGNto6JbxpwsUAQDRlUnWTQagSbFACQT1EFc96nMpJ6MCx/OTfTBivOibyLEB8BW0NW4ebUSlpAH5I2EjWPhu/JUqRVkFI7D9Z37AcQcI38njQFxAxgAHArtNqHPvP9dN6fZODtACCNKAIc6a17NJOAN4Lh/PauDHUKRFTkiF1ITmhFXB6/TP3OBwplhCcryY0+tWrAgzDRQpHNS4JZcMmpT79aS8kDtanS1Ul+7V06Pzsjuz0167VAOTsHPt4yB8MJWkSlALWRQ6iJXGGOO0Cf2W8eOGAKzLMCzU5NbOmxyTsripUmjoogkbJtYhtvOJWFBOYYf86Kqm6JoD2QBI02C8AFDsIgFdGSX+pLr1BStiHwNM16q4hwq4yVTQnNsdR9kkwVz+Hi+h1/T6AUivYz3T5R7ZUDRW1k04vpEIXKA90arZL1LE2cMsQam8orJyOgXjkZT32L7tQx4JPHuQvqFVUyX7FeHfbo0aOTPGWIEfpE7GqZgNlGNqozcEHZAMNUAA/OJAN/gCdE1+Ak8BqgQZ19ZapKR+3otPEAKndXkFqSk6D04hDE1aqBEI1EoN5pqMsjyNIWqw25EZFQUYDA/CKaeByRCf2Ygt7hXy+X2YrxNnpVNwFt7qIMRfRpWBalZy3rJopwTrbuQu1JcwGPhLwRoBN/zvnGNLknnpqJ0WwldpTB1Dk/R2qyPcpRTVDxEOJJGPu2lpwXeZPlG8vDMce2nCYTgV80AmEXCskIlwbOo5tphwxbqUz45jYnosR3S8juT7ZTIvG+lAAhLWqCodrZBX8smS8TWHQ35HrapXxJiOgHlEVxw4kwgIoc0TrLPQUtMEQIhN3bgvgEXKmvkxIlKtiSLAlsKLDiUYK39RUu5PJs/Va5yZZD8rkDt7W2pU3kqThD4E54Stz1SFoH4aep3IqVdXYqj96V4ddV8ZXlPM2RzZ1X2ecLmFXywi6RWya48ris+BEgply810W6WbGd0hvtkN3yC8jWKBgFlnwJ6ZoBntENvQpHt0IcWRmNDNkrv2juIZLEASpfvoJUYzfE/6MEll8sGcTWv+AQE+wwDXsZHj/1yWoZTR60lK3l6IsibsQilvbKtpb91hKgolmyyhQiPgULfM4uKS0YcSlpy5j8/DOSgHQhB7qAGL3BbNPC7XOuiGDUchpJEF0lA6u0H41fGPTHwpzyS+DJPgG2CfiQXJ45UNsY3SjhtEW2LdCSzZOYTEoMYrtllLY0on7GE5lDCjR8HxNONzrboJINOtfQTjXITMPtXZGrTIWsWZNAtJ+ALrx2RAqRcgZNgUnFlPKs2qtCPMyskHtq+P/FponMOJ431glEHc7f6HJFewfYiOhuLTEQWQ2kq6ExVQkMH0rlXiVN5EKkbL5KKqQ4jLWBqMz99O7004cP7z+eHb8yUlffAFjvuAJqt7c7CfTGDb8HfFcTtqjp3+59I8iHkYJzqKhqBeYMYIi6BD3KNI+2siuyQUAJYjT08g13XiQu/g+RmSDvoYvC6eTdH4dvT17F7z+dffh05pIo1VhY5FK1Zn5+cFkV4JBTJo/kop4XFcjZSK2EjUN72j4sjTlxcg/sLoJuvuK7O/E9ev/x4/GRhaqqGt/kOo9DefE2ujBU1D/4DpLD56CVjv8WInLoKm9VzD6D695HxGansAP31xHAEUvMZcESWdRk2MBWZLzhbd9mA0AM7s7QdRJ0bd/xW4jQSt7hyGDYq2mHDmI+IlVT+78AVYSOnU66hWfT70eJw4GJFc4K2LsXE7n7YKtaUzPowK0o0ogWKpK97mCXD4enpy5OFLtxx6Cfr8HKzJftlkIekaaw8l82GENp/HEHABsLrVVGJsJ7vGoDI7fIDuB8VQasWY1Mrf+CfAb1kqyu/4IMrn6r7a+/dxdTEkhhuk2r3celFDRZXGppQugaYBbgOwUToR8gHgcKC6eeL9k66ZNOFc50OdIseZx+Py67WVHoJELT2k6lbj9g8gwJRCkaIwz7ftnY5JokytXpikdxEeFYnI8mxmJOH8L6xcVdjG+iAANjvgNPA9lsXlz4tzu0NnmU0IZYp/2euEbyFEWU37VqxDx86a5LVndodFfI01MPD9HGNprwCLIniuCCcHcQIZJxKlrm2SjVGEe4c6MJgwa3FRPlGNj+ENE0OJvBm8EYXlEQ1AMNgvISNyJbRgpFiDD4IcJfUT3QY3/iscgHPFnkcC1k8X07U9ObfSDgg/EvpVmUcd+euspnbLMC90rT/pGNZFJn7LJVUSLWAtG5d3TVr2/8sgB/DAQiuhnZwd8moOX9mpU06YCWaBxOWxab19vU0eSSeuV2zoWEKIq2005cCPiLruEdVQhGwraLcLI/43lP4lnxmt7M8I0Af9uOxlXWR/ToG59bw4oJp4+bJ9su92goWikybOor9Zz+K6x2UTbZOlm12wJPyy+7rDW7mrOycTw9weAMdAv/7Td2bZ6c+gb3WKzRgUrSNAPOX5hpFyZZ6y5RBFAVW5aT8hED1vMqK5sIT5Z4cUwHTuJWcE5hdqUPJ9iH/QJXHeHTTTrH/ew6lL4jD30nHvB4VMzPQLmhPgzDcfZbMPkZsDLBIE6f+AvcNQBZxXP4DKQcQTyXphkm+5IVpSgglqoMCgE1F9k5L2StNjl8tQ/I4Kps0msKFeVRRVgUaumG/CcfE5boMH598h8QHbUw5Qc5aAi+nQ+ITew6ArOMYErJCrFubmiuYQtwCcuEKFzCZOOawVxSAN1zMjIgnUhlBW749Dl/NKjRgouFEeikxwkVTsDMXh2/Ozl+Fb8eI5NJNn09jl8enh6/PXl3bETCMB++bGzbe3bp5iLciqJm0nhY94KBgEfm8uKHyNUwMKthHLxaoX6qzKNX1FSl9YWio9NR6kiXHJ435R5ylRVV5I7cx8+f6RNc1I2Ocd1zgAuRp0AVsF+EfaoWvw6xvKFEBDopSOk2AE6RUUGDvWzlK1QJtcFKnVcMRYQUCOKkdGnr8OfCRXdrAz05xajKJ3Ru2G1LuWe4YCJOqBiok03N6U6ePpEJ3QXZxlgYatC32F3j2ELOFThhtwGoctxRrjAO2o0bHwdTIjgq+gV8Mf398TcOyouciiqF33tH5azBthNzstMXgnvkdgXM1uRswVqcU+lkY0yvePSjSHPP+cye45ktvnQhrLmPNzsnK2H4fTpW+UKfpXzRe37SlOCK1ZsVyI8gtBRkegni9sKhg7t2wSgT8ow9rBgHyZnMuXID+T47PP0t/nh8+untmdsjHVVxSQvARzOsOW7WYKHChGvUaUC/69Qd/yTqEs3xoSNihijvkBijXJUPi+05zxgMggCocBV+weiZqCF5Aka3mEJ81pYYk0aKroIzDOKqRMVErx3C5LUUxaWibXttlO6z9Z7CmhdU4tYGKKWKayViCEuR6SOlDbtqOipwSHTFxkJRmyrODl46hZ+TEEWWv/d7vff2aog1EETDcsQ0WyzAdXDo5oLOwvzoHDoX2QreYx+0WZSgXyZU9bCGSB/dykQQ0JltuFd6zRpo17AqA+v7FavGmtoYdNiOfMQUnJ+5DhLTa3voXZl9yJFoKUSUqlOUM0YNp6GS7qv97kFp67D0i90npO3DJvzoN6ksWnfXHWIBqhihi5HfUUAmz1tkEZxfwyLE8yWbX1BAyTlfOE7c+0KXT8hCQGXAAR6JWzEUBn6+n+PbPgEf3HQcMQ2QfEgDvnEKNgE1q7giVsYlNAxLIAyGgsTfY8xsWwau3vCbCBDIqTBGU1OR7hefIUQSQG5ekZ/lKbuSH7R7dfT+9w9vjynbTqGUgCPGOfr1+Oi3D+9P3p1ZDqe7KXELKo2TJt40czeUt38M8+LSkxeADOGTP8zqgmeGPF/5WeCfZ4tr9Co9pCkYKdzR+LLJKhZLFKxFEa14RCEeOE/MF+eRPi8vPh2IJRrWeVLWy6IZ4lfBRT86x3QHhFMXm2rOMCrfQOCAtWhUnz/fVGgvccycVS+QKSFgrB2wohzxDNqJKyNwW5tE8fTXQ1jB00+/nw6logTMJqaPPZ1YoccUlZcOP+xYqVPg2tZdC77jLqaAcB2A66yzmieChRRu68j0aRV5hKOBVFnZ5VlM5m5NFiePSHjWMC2u0/khwa2uzmrjyGG0cbLHEWGgOc69dLMHv4dQfIA2EliIGLXtnD5WoAlFoiNtsCBXj/VrG967+gdqJHksIVKn3Wnu3O/gTWYYFEY31hZdo2u4JQRdLyZsOE7Q2BeKCM7EdmUsg6o/mfwp1rzxw37q0lbfAvQ0bnpkWNlv09kELs/aAFjj5A1HWe9o0XakfLhrVF4w3z+eQJsAyy08PEUCQ1vv1FzDnR4CtesbRnlSxrZse0DLPZUHbiQSoh/5qzvGp0xLe/D58qJX4ynjJ5Sdzcidz4qLuapXBemouZYXvF4Mfpm0VAEskKkIMM2LjXaZu+kv3N7V96s0jaEu+KuBl9EgAJl4fh9ROeBXaGl9IvG9GxOjiWV1pzzIrCd3d+8Y5W+Yj1xBmI/lVdKOuG32OOGRaFQszTPgUszVVgTOWJJemHKMebQxv5fanK40IvCIxkeN/Is18C4dj1tR0OhaBiJuK80nXSdSbOJhGohklaHtAktTci+J0DCcDilB/KtCLRCrQp6N9K+goyPXeQnCgr7VTsfqPq/KQCdqrQeuFhUYw1L4msPkmkTmmsjsFbo9PBVYFI1wUukwTVA3RRknC4gVxHO23tBhMIC3Mv1bcXOVmWCLRroWxHyP3D1CZtHpS7mY5v1cRgaTJ9TlFvxXVhXmVi1oMJFrp3vT+hLt1Egfd6rYOWga+CG3KCqYL5824rWQGc2YXwbEz9Ri3RL39oBI/gHyTu0eQE/59YHJDVQZkZV/zVL9BfQabWQDvB0+457e3pKte+x/Zl7aZDQF+cWn8I4IdC3VmiN8UPOiLysaMeBSTGNNA5SZUszqGwUDkX2XTq9jSPPvcQrbfg21u9OnQaPDqbnT4IRGRP0HRQKOoSuT8wT3Mp0EWWeRXeEZQY6luJENdyLmFYivQ5Iiom0KQ4FfF2gxjbH1EUsdcwB+otIWhCy6uTLPGutYtOMdyjLShMrlKNxZF8DjmEbx5GLIOAwCIVFs+g2nnFOqpOsrJtWaobOLaI2JFdS6bejMYJUvrCFbaA/EfH7pSej36gncBhjQzY/zpOSlBtbe4wykPe9Q54UzLzex8U1kUGJ88MwDMtx5iuxyTPsMo28uKlZcWT6YrIwzzmCBQMOKVWAgUs/rUICw8h+PR6NR8Fx3A4RFlx6EB2o6wU++lVQ0q/6skj9525d5xlle9KUd4sDcyLFTL5Z7HirXPNAeZ6iq/jourmyvyNMGbrijoSArWFZOuhiv7hK/By5OXe0DwYNRBrWF6bdTHZhAB/MOZANWW5ffkjBob3HdtUnWt0PmlmDmGW0iubsJ66qaWg67tRPJLQx94tWaIskcWinmFkhQMSCXTJFSuwTQv2qyBQDpLE1x2Z7xg3ZBW0E05aPujKDb09+ZznI6a93Z+bRBZTmW3Mki3FEgiufMN/07pe15o66J57ScpOiyhaMpSfvPU1Hh0qHvXldmvlsYbVhiryCEhQpaW1kPdi9NYbEbR/ZGl7790tjq6drpwGwKYxmbRy9s07AfjQ3De4bXISyL4gIv8uMnvXEvz8htc2uTbioqraJBA2lgKf9HhrjH0qK52uW4di1WFI13JesFRipi+cwDaIUspuxNH4CHbWYdjO2+UHYt2uXD2NeN9nV98P5Y/x4ZyrfYJLtnj0wdhmhltXsdK/Ro7Rx3Hagks+XACMh5tOOzvL00q8GkRzmPdYwLN74bndzItUv4MpkBvSPbNws6qWC7jwguqatM/2/W66TKvjIeLoikMfxqxBCiLyVzaXMRVBjlAcW+Pp0tZDwHzEMbbddF+hnhyFCkP2xRnmYGU+1Jc3empiMVYUAfmq/dmcY0nSmarFQJEi//hZg5eOSN2FSspf+K96NEN7fmtSigHdXNKPbFKOp+up5LK+64O6fezCLzQgCBDt0I0LreBMdGlXHVvaPC2lBPavTwgA28Kzuzps2FGg6Gbx9XxFlP9LWqsbiGFkbDQmQsTuEjgB9TNHiTFNUlAhwwGhsqX4kxwpLNDuRnfl3ETKfY2l4Xt1o8d1GjJzBvPHkbTncmwsTZU1GHTnCxo/5SIssP2llH5LIcHFrG8MIMUM0N7yGP2vJEujw/EfLDALlzeIK3xFQ5nv9gW3HmkbKdVcY5Yc80oNB0voyX1yWe2aoz3IJ69/4sPjs+pS0mXseECSA8IkdiaKaAhAWv8e7AmPtzWGhEi8K3uZIGL6+RKSTBWSYIXXakWoJXot4SEDx8JRB7c/zu+ONha/vrYdVf5n26eBSMeCsWTgZfcXNqNAfObA/j457JkWoKhdyDCHKnsbbru26VdqRdWMwnYCqC+kCY/rC8ipmYEGpRQDhw16ypsnkt7QG9FxYkmrg/OuKPOvzPf/2381IexzkM5Z4+u2JzzkQyGxi4mvju70D70Pmn4N5/Op7YCrP/RATn1gDPm9JiHRx+OPGHBpiPghUdzYqh8/ix5sXHj4eO/IMP4oujmME5cPD58Ojo+PSU2lpILh4NnKd26otMC948hTzOd7/l4V/O5f7tI6v/Gb/QrHbwmkzsHDpPnzveacivPt43LzwOnDd9r30b4qFgeEDClI3bgKtPfM/XamIy49TAyx04x2AyKkwSAkEPiLQOyUHojMCi6hxizapthluTS7wFA9P4uK2aOrNr+fc6QudjOUJqGKvyp/OapOSAW5U/HaykhR9niA/8POXK1vkTyP3nYDDA/0Ljf6425RfsOrji5dQoabL8T1siZRcXj/50bqD5LQyAySAxbfHEFb56tNT9NByO/wJfHgnulgZ1AnNy3xXO4ZuTwEk2TZEXawgVHKUfA2deFXU9oJ2sRlyXhZf66euieNZulX3lN2XNV0m2xtyPOrWmuNk9Rc3Ai1FmzFlm58vW3zSB1/Nkg3dUOVQpv6ZSAuB6vKGLMW7ks6RyTvWFNDXZXTxUkCZVqs50DzHTK+/ZVhfD0ua4vFaMl88Yw7/AoXCLYI11eOS+I4uY/EvH71mCtTWvnyki6Cke4tSTlpDTX3TBINH4cy4vHL475Lw8OKJAA9zCGmcjPC+HNmIwdAEoTQZoqK6CnXlKf5MjWJUZcGaUA6s1Rh9Qt4Js47UUCa2mc1SskhlqLUypcfrB2Cvxt2Sw/D6Xf4xG3whIe/dUQ4QaZcZIla15+TWtsthG7GhZ7vNeD7N0uE5Bb5v1NRBkcB1T84jjznoaDlKWKoMxz+N6DQH7fb40beyCEVF5Xf5nZobVugGmMhqSm6jdatr/CPTnlvNuoQYvBGLIO1J8kzKSfyJneFidb5DJPuBT5aWMF3IDySNXGBpJ9Lv+ipAgSVJi6V2cCKBYxADI5akb8Hu38S+E4H4Z2VZw7jGtxNfB9XdAGAxwFm4Ak0jAnEX4F2rixRgi+iylA3k7+5GDE+CRITxXqDDg9mJnp6YoBxQV6647G6+TqwF6LAP0WPRIEtWR6BhBV6I2dq49XcgxFOSJIqKKoVzR2kWTZIi/THlbCq7IE6YZ2FGECLyElaRiAjN0lltmyZA2zYyNMoCrHuwNsWRoPvqqMq2FuFjIsBcf784p+GEbUSPkI2nh+Po7g8YdAaFZRdceQouLAI5FtYBdTH9XJY5hSnGM0hLHbsilZu9/AVBLAwQUAAAACACUIGRdlMb/Ul0JAACYHwAACgAAAHRlc3RfZjEucHm9Gdtu2zj2PV+hzQtlrOpG7k53xoEe0sTtBjuTFk06WKxhELRExZxIokJSid3A37P/sV+251CSJduy4zSLMWBbonjud+r4+PjmkScP3GFhyHPDspA7OuQZU0JqJ1Yydc7OR85H3/mQFDxXIjPOf//jnzh/dbRRIjSOLvI84SnPDEuccMbDO90/Pj4+EmkulXFCmS/q6z+0zOprPSuMSOo7w9M8Fgmv74tMGMO1ObIc5MzMEjF1qodf4LZ8UG/rpzK8qx/D7nBWI2Ihp7HvMO3Mjo7ChGntnK0kvQFQ7a6Q4O0507w3PHLgE/HY0dx8y13Nk7haxA/e9k2aBzXXAInUmFpcCMVDI9XC7a1vV1KaABl3a+h+xlLeW1EynKkL+ZhVxFZE+mHCWVbkgLC1Vxv60ac3Jz7Vi3QqExFSxWOuOEhFgRmzoFGRwzIzfJP9WCrHyDueaUdkjjueeGPCSPnr2e/U/k9aMDWczLliRsgMQcdPROZkSBR/4EpzsvTqBWmALk14bACTNjzXZOgPlt4avu1PBQ4qNVxRfg/ADwzcDtZYgz3iIFqLmn5kOc2ZUJosJ8MtEorrIjHBrA+uy1WuuKHafSKlBsiw/PfISjJYG69uJsveFkJrGXAkrsylvsy0dSW3JOMlQpu9IKP7giX17h9mqiFhmL7TIN0tz/Axd2f969Ho4np8MumdbhNOeOZakJ737n0LS2AXAeh0C2TW52gFxI0cMxqLOY+olpA24GFeTNH9EN41vV5vTHQoFQeH8jtddkBvFctnFOIFXJsawTUtMsVZOGNTWPjOldx0WRNkfG7cufXBOfpet8COiJ35mMQsFcmCTIKAfCJtITEZ9SPOc7xwW4bKgydyJ7II3MlyB56XyYij2skZ3HyA7zl8L0CstnUJj27trtW2Ewyks3K7vf5gwbwBXp/X1xtYIps2OJD/yBLNMWaYMsAMIrqVLIHLC7JsJBmTXEnQVgpC5muRHbIsEhHoxUZow/0FaTGGdxuhIkEDaEMqgAvAX19PIBVk+pErMnwimIjJcEXCI6HUwObAIyv7keGNKvhyebTX/9d8SqLPQMYoNFL7cnZ93TJaPq50PAnGk6Mf4BbyW8Xmlcz4GqdW2S1WX8FmC9RiBdDKwa0/uTkAN5QnnZHxjooMyImIysLkBaQEKKcp09vR0O398F1zBWnz+1NLI3V+GU+WS8/ZocLONH2AumvkmKtfi2PsT16PA1YhaygGtWe53ChmLzH15dXvZ79eXtDP326+fLshnbb7G9TfnDPDpiIRUH2xImLepprzSHeVYHxQZjJrwHX2jN+2MW7tnZrB1tpzIkUCIse4xu95zc2gtw4394Pxs2m97BqQX+NP1sEHLwMfTA7ket7met7mugVzJTfBumOjhapjgw8bOs36E51CQ5iIDLpIJb/zjELuo5mE3oapbNOsUw5icrASJOcH6KGxLwBPCqGSUD1jg5/ek8lhhXu74Nnt2/WtnYNY8Kwd1vODmXG1wr3ugPsxWchug5TWYAc0CruhSz16e/W4J/WybOHelaG1TtNKbZ+4REN3bLtTiCP4w1Y+waYVsskOX3hPZzyJIDVTlgsa8UywBF0hBTAF15ve8CjMrM3cVyY016AIHEC0vgAEEMJD9ABTckiQwBugQLwdXtwlNNZblyWJCwmxlGryl6BB1YTeDpy9l3K84Si2ZKHqbDZOWSZiDLNxK1WTT2/+8fnNiQ+KrnwXllD5lt1hS+5VXzN8Wi67y+TfaaiYnlEW2ykBHMTQ0mOoHT9zCU01kMWZzOBDbLZTfqB5LrEhV0UO3dhIKam2xFVF1lKft5ruPC3SwroQVI6kzUuAFmq0DAgoNHzBCvItgSVN3sbkjH68/Nfogj41BJZkd5xgMw8McRZBPoImSbkV7rfEqkX3ceJOwKGxF1+hweEs2C3InsC0oFXZLUcYLJDtQULJR0xqpVWqLQfyuF9ORNzz8OpJtQq/9W6bxHBDezCqkvEuciVL0wX4VWtW/yGtVOnqIEqdHv0znRYRZgEsLo8QyjQRqTD64IzywYJX/rrHRVM2p9gLg+pU4O9Joe6mc/b6fC7wtGQzXdjTlr6c/gEDjDvzyNdvV/Tm8rcRNEv0enT++erimnhv/I0weq0kDROQNV4VSZW8gGbLbo3EHTb7ZdWs1xnPWq9qIiUkJpRB05iJhIaJ1Dza6uP3dACNG7Noc2y14/spPICNrVAIcMVvrzznOL/j4cpK1zvyOGB9CTetOlpxNNhc/tPYsq0U8tBMy+P2mQqMlNUZ0rerr+Crn64u/z2COfuFHK4GByDltaqepfzMiAKDSYd3+SdVXUOXKrMoiJ8C5yK7hdbD2NOCTX86JFximUT8B2pPHJSQG0FyKpW4DeLuZBr3H5UAw5XruBP2gRtANzetyRFvSj7Ul/4hvQgW6FvAu2g8BNQSLygKXzK5j4X2mU8tUrue1YIZ6L+DvBQMr93eaV4hs7f4sxKnPA3wyMezf47+n2J0eIZP4wIKhOKhzEKYNa0rU6wZB0+apZ/gapeHgEyQk6BjSpkS37l7SAUEGPB0CX1PGaXMYMDb5sA/+fk5QOz+qO2rG7CTfV1Ikbrzil7VAdgRySKDGKFlj0mrZrhvz5Axjx/CTClFDuvARUWpvHsBoed6GYS3cTfpee/2N/VznHsgeULMb7FQoej0kwHVGcv1TBrK5/gqhOJPfTyBeUWrkM6gkf4z08imhB1e7yl+Xwhs5iuxy/a5rYaGjwjkFZmNgTYzqbjFghq1CNvXTX2sDUZxXtNqwff2HQNux6a3ttjG0yAK49ug6n0xq7h1voHQjSEZ1gayKYfsoQ94bAmrZmGsYmuzMPCCr6FKa9qXTLM+pXaJ7nXGDsQsigSKAUFQd4plOK77WPPWbzUVm0XOaSp0ih1hNR8f2MGu19NmHo6lJN540nvR0Sc2A5uTf9kF2Fl/Epy8jJ31cwzbd+zUBRSRHDoLXuoOk3SaF+CINuDA7NooBhOhfuULjvUuKyCf+mkB9B6EFqb/4JPehjKeOYoptVO1J9a5NKSz4Bf4PHPm1jozRcXsOSC/3/3y5X6tP8MZZSZz7M78zi39XOYuaQRuEZm3Xwvc738tsHozMiKTg15k7DsyvvfmG0fG55+/QktpT4vBZpTi215KwVYUeliRUUqGzdtrWHAhm0wliLMIBr2j/wFQSwMEFAAAAAgAxiBkXUQWUQzYCAAAMBEAAAkAAABSRUFETUUubWSNWM1y2zgSvuspumouiYaiLNvjmdi1B8WWHc06tspy9rAXERJhCiFIcAHQiat8mNM+wO4+wz5YnmS/BkhZzuSwB9ESAfTv1183/BNNz2d0OaFvf/yHLtVXmdN74aRWtaQPwtbSOXo8SCeDwXB4oT4LLepS1JTzR7WfFTUiF3RMN+ZRVmtp6fDg8CQdDmleK1JVo2Ulay+couFw3QuejoKm4TDhty0E4oAoZE3TOVWizpVVJLxoCYseOli0Fp+FFfVYPhrdQl4pN9Lmwok6HQx++ommjaAnURewrREe1mlRsujBYERXspZWeGMpl17aStXKeVVS3grI0a2whTiFLUvY8cZDi3swtmKrnarWRqvyLZt6xcuFFQ/B/8Kq/G0K6RcQZtWajWqsGFlZ8As+3SAgTsr8lCaHlOXyUWrTcEAy+plOKHsUWuXCK1NnYW9vC/2Fjk7It4VwYz6f0oWsC+jkHzT59fBdwoH+NaHjg3dH5I0XmiYHv7GQjVkLjsmIsq3U+ci00HZ5TG+ODmh+8UrNWXfy5OAtvPMwBSGBI8IntPslNiIJ/vZvdlm/nAyHKd0HER7ypKdGaeMDpDh6W3p3wjrbWpWQKL1oFGMNESj2scGIOgo7v4tfHr32CiFRtfNIjApZ4ZjAw6XRj/BnGmXTG8jL27IFCAz83M8fMVJslzckkGXlwiKtEkloAzaAjEY5sSXgGFhVjqx8kFZCaULaFPxewGBFyw/T0eEvJwlttnJTNkbVHng1FbvJKgDYVqstvq0BQ3hlnHQsHxulbRsYxoKsoI2pKuVhBCIRjr5IZAcBG1fKWlhlIBQgVpVgqbqVDQtDNEf3B5Nvf/yXv2D3z3TI2rGxdLzTi2otYMgZTY5hgCMDM4E4x9IXT35rEPo+Gglh55Og88UnpEvUqCjUUD2eLubjtRJY0XjUe2Jj6f2+owVFV8YUWtK50WJNWn5BKX5YDAaTlD4hN1vKAI7V5WTVs8wq7Fwxx6SqearXGcJgQ+lsRIlSho7DlN6jlAFo7xt3Oh5v+ExqpZPCbrZpEXSmiOU4SxiDQN5weKlgx7d//os+NdqInGrj5dqYEqAdHKW06LbdtbVXVdyJ7yS05h3H2CGYJOAWSqhpPcO2rT+3Jb/K7vCIIfUOSrPbv/Iz1NMKmC5XwntZNd6dcmFmERcZHL++PZ9erxbT5TJLB7/0Yfn7fNGRXeQw5cUXYVlTbzYgCKCilooCEC0R4q3wV4t7vPFtSagD9QBrAa0tHjrKWTN2RuHJ6A38HYsXgFRdlqbIPRJ50yki1+aCIwN/nyTAwla4NrB7aXIJEowqIxU4uVVsU8cPFYNPM/HGitWBB0UJcuhBE6mdSUCUXBfgdcV1ntKl0Tm02C4lLhIEyiQaukZ10hYVBc8ciFzDzOAHF2a5VfYMlvXRjDayD6AarhmOLugqWMkRaPqlULAW9e7Rgfr64Ah3WL6w6lF+j3TGKKuJrr1ZnI9jNb0dDLIsA31sB00sr1HFBOgZKTR6DIhZPUz6VbGR+JU2T+w1jUbWGKYDBerDSi2//HljSPXT/7cXeauE/fFm2MltfZ4Dp8g9aO4UuW0CSyGyGQzKdt20DVHvCDnIYkA/ctzIiQq9JMIutlfuCF272CrPhKxFgej/DoSC83LJuwPRtmtOooAOzgnHIAA3tg/P4jlFCagYrkguDECh5LLh9L6aMNbCtgkVbS0iF1lZgtYVZfteZzuaTwC84A0ga/K+eGIRbgEGHHUx6wt4Dagzcp3umh+4q+BOg5ddIfKgkU2Ox5PjjLgwsj7r2SlxuWOykDxUwOxA2f8GX4f54Z77PE0DUSQckxjJ347oTbZEyB/RHMdHJ8wvS0K3ooPxhBklu3q9eBUWJxNeDYKXaEMC9K5lXkg75uCCdKVDB+PkaaON2yOOcEShc3EcNpYT1FfZqzblJKaxttprVdHBsz5JsXVwgRfcVzgbwaXQc/oJBeu2dYBIQEI0MTZPa55kueOxPuM4y30IqZKasI5gAmandHBGP+5NsUa33RB71xykYSgN8+Y+XTqhczP2wGfgJiQ1Zn3phW9dMGkNxnJMGB5ZHwyeaeoaWdJzv+d58DwajcIHi5hwXkZd7ol7WIm02zH38wsuOg2MLOZYgDQ0wCfxljqRXGQsLFJhpXioaTjEwDtLmt1czG+u6G62uJ6fT+/ntzfh5IduBoRmgSDzJPhMN7f3q6vZzexuej+7OANdbDhEhfCSK1xJnnz+0XJrA4mzcpb0nqYgAw8upDGd04zncB5dmV6iyPnHxfXs4+wGQsOJxd7czqN94P0izOIaFcgp7q2LAu5ny5ezP2JnzdTk9pvXnw4uAbm1+frC/7tLSwgszAgE44G8XZB/YP84vFlOL2er+9vV+fV0/hHiQZfvQ6ZKiXTANmbM0tTeGk2xm+1cQtiQTq6esCF2nYYHzchwsnpQGqmnB4UjSQdM5Uyov9tlivrVrW230XBAnqkZDoXKacTOpW6Me/Em3s48RlDH16lWaDDvjutwsO/IuAe9xKmABlmGqxVLjXny6gGMBDj3TTsQGKu2guszTogJ4bFRXKBJ3IhbDc+9uD7koP0PHdnH6RlC1vgWRgC+j5XxptSGqt3dE2I4Ih1jtxphrBXatX1hc/tmiUUfzrro8uC6sAmgpkmZcdZ8pwz+vx+fU83ieM4FN6B4+5CU+Fs1LB+jl96r+3hDcN1oBkoEnhP8tV7sjzuuAx5A0ELc3WzZ84iFE60N/WHXmbNw4y76i2myRxkJGSs2Wu7dTRLujNi6f+NIvqPurJssdsJxHeDKbkDHGxluBq5t+n6p4+gaDu76ozYbocfx9FFoxgKWC51E5nev1YPtG1M72NlNGQnCgZvwJkrFrDvdqV/ds3HXpkj9V8/NCqt/C+Zvwv13txR096MEJqaev7Uphe7lzr7KTeCe1Z1sjPVplceDPEiW/D+BkL5SYmxv+VcRMFCE87i84e62/PRx+UohNztul8J6+YBijVMyDybMsox7vor8D1BLAQIUAxQAAAAIAJQgZF1cx3o3fyEAAEprAAAJAAAAAAAAAAAAAACkgQAAAABhY2VfZjEucHlQSwECFAMUAAAACACUIGRdlMb/Ul0JAACYHwAACgAAAAAAAAAAAAAApIGmIQAAdGVzdF9mMS5weVBLAQIUAxQAAAAIAMYgZF1EFlEM2AgAADARAAAJAAAAAAAAAAAAAACkgSsrAABSRUFETUUubWRQSwUGAAAAAAMAAwCmAAAAKjQAAAAA"
work = pathlib.Path('/content/ACE_F1') if pathlib.Path('/content').exists() else pathlib.Path.cwd()/'ACE_F1'
work.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:
    if any(not pathlib.PurePosixPath(n).name == n for n in z.namelist()):
        raise RuntimeError('Unsafe payload path')
    z.extractall(work)
print('ACE F1 fixed baseline | runtime:', platform.python_version(), '| root:', work)
print('Checking 12 blueprint acceptance tests + 2 supplemental tests...')
run_tests = subprocess.run([sys.executable, '-m', 'unittest', '-v', 'test_f1'],cwd=work,text=True,capture_output=True)
(work/'F1_Acceptance_Test_Log.txt').write_text(run_tests.stdout + run_tests.stderr,encoding='utf8')
print('\n'.join((run_tests.stdout + run_tests.stderr).splitlines()[-22:]))
if run_tests.returncode != 0:
    raise RuntimeError('Tests failed: no claim of success.')
for cmd in ('run','verify','summary'):
    args=[sys.executable,'ace_f1.py',cmd,'--root','evidence_local']
    result=subprocess.run(args,cwd=work,text=True,capture_output=True)
    (work/f'F1_{cmd.capitalize()}_Log.txt').write_text(result.stdout + result.stderr,encoding='utf8')
    if result.returncode:
        print(result.stdout, result.stderr)
        raise RuntimeError(f'{cmd} failed')
report=json.loads((work/'evidence_local'/'reports'/'metrics.json').read_text())
if (report.get('total_task_attempts'),report.get('total_pass'),report.get('held_out_attempts')) != (108,83,0):
    raise RuntimeError('Unmatched baseline result — inspect; do not mark PASS')
runtime={'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
         'python':platform.python_version(), 'platform':platform.platform(),
         'google_colab_module_present':'google.colab' in sys.modules,
         'project':'ACE F1 fixed baseline','note':'This JSON is a self-report, not cryptographic provenance.'}
(work/'runtime_info.json').write_text(json.dumps(runtime,indent=2),encoding='utf8')
with zipfile.ZipFile(work/'ACE_F1_reproduced_evidence.zip','w',zipfile.ZIP_DEFLATED,compresslevel=9) as zout:
    for f in sorted(work.rglob('*')):
        if not f.is_file() or f.name=='ACE_F1_reproduced_evidence.zip' or '__pycache__' in f.parts: continue
        zout.write(f,f.relative_to(work))
print('\nF1_LOCAL_PASS | tests 14/14 | attempts',report['total_task_attempts'],'| passes',report['total_pass'])
print('ZIP:',work/'ACE_F1_reproduced_evidence.zip')
try:
    from google.colab import files
    print('Downloading F1 evidence ZIP...')
    files.download(str(work/'ACE_F1_reproduced_evidence.zip'))
except ImportError:
    print('Not Colab; ZIP retained locally.')
